## 04. scratchpad

### 1. Data Upload

In [455]:
import pandas as pd
import os
from collections import defaultdict, Counter
import re

In [2]:
def upload_to_dataframe(root, files, num_lines):
    path_eng, path_pol = [root+f for f in files]
    data = defaultdict(list)
    
    with open (path_eng, encoding="utf_8") as f_eng, open (path_pol, encoding="utf_8") as f_pol:
        for _ in range(num_lines):
            data['eng_text'].append(f_eng.readline().strip())
            data['pol_text'].append(f_pol.readline().strip())
    return pd.DataFrame(data)

In [3]:
root = "../data/opus_opensub/en-pl.txt/"
files = ('OpenSubtitles.en-pl.en', '/OpenSubtitles.en-pl.pl')
df = upload_to_dataframe(root, files, 600000)

### 2. EDA & Sanitazation

#### 2.1 English Non-Ascii Sentences

In [4]:
def nonascii_list(df_series, is_pol):
    if is_pol:
        pl_chars = set("ąęćłńóśźżĄĘĆŁŃÓŚŹŻ")
        requir = lambda x: x.isascii() or x in pl_chars
    else:
        requir = lambda x: x.isascii()
        
    text_all = ''.join(df_series)
    unique_dct = {x for x in text_all if not requir(x)}
    return sorted(list(unique_dct))

In [5]:
print(nonascii_list(df['eng_text'], False))

['\x80', '\x81', '\x82', '\x83', '\x84', '\x85', '\x88', '\x8b', '\x8c', '\x8e', '\x8f', '\x91', '\x94', '\x96', '\x98', '\x99', '\x9c', '\x9d', '\xa0', '¡', '¢', '£', '¤', '¥', '¦', '§', '¨', '©', 'ª', '¬', '\xad', '°', '±', '²', '³', '´', 'µ', '¶', '·', '¸', '¹', 'º', '¼', '½', '¾', '¿', 'Á', 'Â', 'Ã', 'Ä', 'Å', 'Ç', 'É', 'Ê', 'Ì', 'Ï', 'Ð', 'Ñ', 'Ô', 'Ö', 'Ø', 'Ü', 'Ý', 'Þ', 'ß', 'à', 'á', 'â', 'ã', 'ä', 'å', 'ç', 'è', 'é', 'ê', 'ì', 'í', 'î', 'ï', 'ð', 'ñ', 'ó', 'ô', 'ö', 'ù', 'ú', 'û', 'ü', 'ý', 'þ', 'Ă', 'ď', 'ę', 'Ł', 'ń', 'ő', 'œ', 'ś', 'Š', 'š', 'Ј', 'С', 'й', '،', 'أ', 'إ', 'ئ', 'ا', 'ب', 'ة', 'ت', 'ح', 'ر', 'س', 'ش', 'ع', 'غ', 'ف', 'ك', 'ل', 'م', 'ن', 'و', 'ي', 'َ', 'ُ', 'ِ', 'ّ', 'ْ', '\u200b', '‒', '–', '—', '‚', '€', '™', '─', '☻', '♥', '♪', '慹', '拢', '檛', '鈥', 'ﬁ', 'ﬂ']


In [6]:
is_ascii = lambda text: text.isascii()
maska = df['eng_text'].apply(is_ascii)
df_non_ascii = df[~maska].copy().reset_index(drop=True)
print(df_non_ascii.shape)
df_non_ascii.head()

(4595, 2)


,eng_text,pol_text
0,¿You had never seen A crab suedes? Them there ...,- Nigdy wcześniej nie widziałaś raków?
1,"He hears ¿, where you live?",Nigdy wcześniej nie byłam w lesie.
2,"Boys ¿, they are there?",Jesteście tam chłopaki?
3,- Te atrapé. - No es gracioso. Your he catches...,- Ale twoje mina była.
4,"Your also ¿, it is not like that?","- Tak, niezła jest"


In [7]:
df_2 = df[maska].reset_index(drop=True)
df_2.shape

(595405, 2)

#### 2.2 Polish Non-Ascii Sentences

In [8]:
pl_chars = set("ąęćłńóśźżĄĘĆŁŃÓŚŹŻ")
requir = lambda let: not let.isascii() and let not in pl_chars
maska = df_2['pol_text'].apply(lambda snt: any([requir(let) for let in snt]))
df_non_ascii = df_2[maska].copy().reset_index(drop=True)
print(df_non_ascii.shape)
df_non_ascii.sample(10)

(1227, 2)


,eng_text,pol_text
328,"Laughing, happy.",Šmieja sie. Ciesza sie.
99,Death!,...jest mi ¶mierci±!
811,Are we going to let a man like that throw mud ...,"POWSTRZYMAC SMITHA Pozwolimy, by ktoš taki obr..."
143,He was going to send for you As soon as you go...,Zamierzał po ciebie posłać jak ​​tylko stąd wy...
951,There is a legend he can drink two bottles of ...,"Istnieje legenda, że ​​może wypić dwie butelki..."
157,The JurgenStrasse isn't it.,"To jest JürgenStrasse, prawda?"
724,Every aspect of this matter was dealt with in ...,Wszystkie sporne kwestie zostaly wyjašnione na...
767,- I've seen filibustering...,PAINE'A - Widzialem juž takie filipiki.
319,"Come on, boss.","Šmialo, szefuniu."
46,This is blasphemy.,BluĽnisz przeciw Bogu.


In [9]:
df_3 = df_2[~maska].reset_index(drop=True)
print(df_3.shape)
df_3.sample(5)

(594178, 2)


,eng_text,pol_text
460399,"Whenever there's trouble, you're always on the...",Proszę wejść.
422385,You talk as if you wanted to see her tried for...,"Mówi pan, jakby chciał ją zobaczyć sądzoną za ..."
116876,"Ask me anything else, anything!","Proszę. Proś mnie, o co zechcesz."
42758,And I'm going to take you in my arms now and h...,I zaraz wezmę cię w ramiona i przytulę mocno! ...
33941,It's Elizabeth!,To Elizabeth!


In [10]:
print(nonascii_list(df_3['eng_text'], False))
print(nonascii_list(df_3['pol_text'], True))

[]
[]


In [11]:
df_3.iloc[74182]['eng_text']

'The next time I...'

In [12]:
df_3.iloc[74182]['pol_text']

'Następnym razem...'

#### 2.3 Short Or Empty Examples

In [13]:
mask1 = df_3['eng_text'].str.len() >= 3
mask2 = df_3['pol_text'].str.len() >= 3

In [14]:
df_4 = df_3[mask1 & mask2].reset_index(drop=True)
df_4.head()

,eng_text,pol_text
0,"Previously on ""The Blacklist""...",/W poprzednich odcinkach: /
1,- You want to call your daddy?,- Chcesz zadzwonić do taty?
2,"- Yeah, I want to tell him I'm okay.","- Tak, powiem, że wszystko w porządku."
3,Okay.,Dobrze.
4,Lizzy... Be careful of your husband.,"Lizzy, uważaj na męża."


In [15]:
df_4.shape

(593894, 2)

#### 2.4 Dialog like sentences [starts with - ]

In [16]:
is_dialog = lambda snt: bool(re.match(r"^ *- *", snt))
mask = df_4['eng_text'].apply(is_dialog) | df_4['pol_text'].apply(is_dialog)
df_dialogs = df_4[mask].copy().reset_index(drop=True)
df_dialogs.head()

,eng_text,pol_text
0,- You want to call your daddy?,- Chcesz zadzwonić do taty?
1,"- Yeah, I want to tell him I'm okay.","- Tak, powiem, że wszystko w porządku."
2,- 45 seconds.,Szybciej! 45 sekund.
3,It's okay. It's okay. It's okay.,- Już dobrze.
4,It's okay. You're okay. You're okay.,- Odpoczywaj.


In [17]:
def fix_dialog(snt):
    return re.sub(r"^ *- *", "", snt)

df_4['eng_text'] = df_4['eng_text'].apply(fix_dialog).reset_index(drop=True)
df_4['pol_text'] = df_4['pol_text'].apply(fix_dialog).reset_index(drop=True)
df_4.sample(30)

,eng_text,pol_text
313445,So you and the old boy cross swords?,Więc posprzeczałeś się ze staruszkiem?
139898,See you all later,Do zobaczenia.
171937,Love is a strange thing.,Miłość to dziwna rzecz.
445872,What about the poorhouse?,A przytulisko?
223879,Halt!,Stać!
531417,I'll get rid of them.,Dlaczego nie mogę?
1364,"Yes, of course.","Tak, oczywiście."
163491,Try and come to an understanding with the boss.,Spróbuj się porozumieć z szefową.
31358,You're not going to see some young lieutenant ...,Nie masz dziś w planach randki z jakimś młodym...
136937,I couldn't marry a dead man.,Nie mogę wyjść za nieboszczyka.


#### 2.5 Broken Dialog like sentences [somehwere has -]

# Do naprawy (lepszy re przed tokenziacja)

In [18]:
broken_dialog = lambda snt: bool(re.match(r".* +- +.*", snt))
mask1 = df_4['eng_text'].apply(broken_dialog)
mask2 = df_4['pol_text'].apply(broken_dialog)

In [19]:
df_4[mask1 | mask2].sample(5)

,eng_text,pol_text
407931,Get back!,Wracać. - Ewakuacja!
508312,Would you take a note in for me? - When he buz...,Zaniosłabyś mu liścik ode mnie?
492639,He's coming. - Who?,Kto?
127576,"She's gone, sir. - Gone?","Wyjechała, proszę pana."
329368,"And maybe in a flash, like a bolt of lightning...","Nie przetrwałby bez korzeni, które sięgają głe..."


In [20]:
df_5 = df_4[~mask1 & ~mask2].reset_index(drop=True)

In [21]:
broken_dialog = lambda snt: bool(re.match(r".*-.*", snt))

In [22]:
df_5.sample(50)

,eng_text,pol_text
424250,"Oh, how I do hate living in a London flat.",Jak ja nienawidzę mieszkać w londyńskim mieszk...
203754,I can't.,Nie mogę.
368225,HeilHitler.,Heil Hitler.
428530,"There's a $10,000 reward out for just the kind...",Jest 10 tysięcy nagrody za ten rodzaj informac...
565153,Yes. Is there something I can do?,Jestem jego żoną.
301504,"Yes, yes, I'll do it.","Tak, tak, zrobię to."
537393,German victory is inevitable.,Zwycięstwo Niemiec jest pewne.
391344,You hear that?,Słyszałeś to?
504670,"Furthermore, she knew precisely how to guide m...",Pomogła mi wyjść tylnymi drzwiami.
51847,He's probably under one.,Baravelli do mnie.


#### 2.6 Different Last Case

In [23]:
mask = df_5['eng_text'].str[-1] != df_5['pol_text'].str[-1]
mask.sum()

57558

In [24]:
df_6 = df_5[~mask].reset_index(drop=True)

#### 2.7 Text length difference

In [25]:
mask = df_6['eng_text'].str.split
df_6['len_ratio'] = df_6['eng_text'].str.split().str.len() / df_6['pol_text'].str.split().str.len()
df_6.head()

,eng_text,pol_text,len_ratio
0,You want to call your daddy?,Chcesz zadzwonić do taty?,1.500000
1,"Yeah, I want to tell him I'm okay.","Tak, powiem, że wszystko w porządku.",1.333333
2,Okay.,Dobrze.,1.000000
3,Lizzy... Be careful of your husband.,"Lizzy, uważaj na męża.",1.500000
4,I can only lead you to the truth.,/Mogę naprowadzić cię na prawdę.,1.600000


In [26]:
thres_left = df_6["len_ratio"].quantile(0.05)
thres_right = df_6["len_ratio"].quantile(0.95)
thres_left, thres_right

(0.75, 2.5)

In [27]:
mask = (df_6["len_ratio"] >= thres_left) & (df_6["len_ratio"] <= thres_right)
mask.sum()

472209

In [28]:
df_7 = df_6[mask].reset_index(drop=True)
df_7.sample(20)

,eng_text,pol_text,len_ratio
255829,I've got about $500 left.,Zostało mi 500 dolarów.,1.250000
375607,Sorry.,Przepraszam.,1.000000
115059,Mr. Waldemar!,Panie Waldemar!,1.000000
149747,Hurry.,Szybko.,1.000000
273679,I see.,Rozumiem.,2.000000
339633,Even if it is about Hitler?,Nawet jeżeli dotyczy Hitlera?,1.500000
139800,You'd never sell your old stock.,Nigdy nie sprzedałaby pani starego towaru.,1.000000
12781,Look. My family.,Moja rodzina.,1.500000
414887,You see me climb in? Sure. Or I wouldn't sugge...,Inaczej nie zachęcałbym cię do wyjścia.,2.333333
276094,I saw the tunnel that my pal Lefty Matson dug ...,Widziałem ten tunel co mój kumpel Lefty Matson...,1.153846


In [29]:
df_7.sample(50)

,eng_text,pol_text,len_ratio
173037,Twice I've seen you dead in my dreams.,Dwa razy widziałem cię martwego w moich snach.,1.000000
284328,It was pretty sweet of him to come from New Yo...,"I to miłe z jego strony, że zechciał przyjecha...",0.866667
162221,Reverend Ethan Wilkins! Reverend Wilkins has c...,Wielebny Ethan Wilkins który przybył z Marylan...,1.600000
160195,Right.,Dobrze.,1.000000
106869,Ferryboat.,Prom.,1.000000
86412,"If it was anybody else but you, I'd chuck you ...","Gdyby to był ktoś niż ty, wyrzuciłbym za burtę.",1.222222
365955,Shut up. You're a cheap crook and you killed him.,Jesteś oszustem i go zabiłeś.,2.000000
200980,They can fall out of love.,Mogą się odkochać.,2.000000
427879,"Well, there is some ale in the kitchen if you ...","Mamy w kuchni trochę piwa, jeśli chcecie...",2.000000
295928,"Remember that, Tokyo.",Pamiętaj Tokyo.,1.500000


#### 2.8 Sentences starting with * fix

In [30]:
fix_star = lambda snt: re.sub(r"^ *[*].*", "", snt)
df_7['eng_text'] = df_7['eng_text'].apply(fix_star)
df_7['pol_text'] = df_7['pol_text'].apply(fix_star)

#### 2.9 Action Comments

In [31]:
def mask_distinct(df, col1, col2, chars):
    chars_escaped = "".join(re.escape(c) for c in chars)
    templ = f"[{chars_escaped}]"

    m1 = df[col1].str.contains(templ, regex=True, na=False)
    m2 = df[col2].str.contains(templ, regex=True, na=False)

    return m1 ^ m2

chars = '[]()/\*+#$"️'
mask = mask_distinct(df_7, "eng_text", "pol_text", chars)

df_8 = df_7[~mask].reset_index(drop=True)
df_8.shape

(467550, 3)

In [32]:
df_8.sample(50)

,eng_text,pol_text,len_ratio
198320,I know.,Wiem.,2.000000
4018,"It was very good advice, indeed.",Ta twoja rada doskonale się sprawdziła.,1.000000
421037,Truffles and champagne.,Trufle i szampan.,1.000000
89116,You sent for a manicurist?,Zamawiano tutaj manicure?,1.666667
107639,Please.,Proszę.,1.000000
301349,Where do we go?,Dokąd idziemy?,2.000000
67230,"That's a short jerky movement, like this.",To taki gwałtowny ruch.,1.750000
416361,"What's the matter, Mortimer?","o co chodzi, Mortimer?",1.000000
401008,Come on.,Wstawaj.,2.000000
464452,They said I'd never do it alone.,"Mówili, że nigdy nie zrobię tego sam.",1.000000


#### 2.10  " ` " ---> " ' " & del " ~ "

In [87]:
wanted_case = '`'
mask = df_8["eng_text"].apply(lambda x: wanted_case in x)
df_8[mask]

,eng_text,pol_text,len_ratio
214145,"Revolution had broken out, her diplomats sued ...","Rewolucja na tyłach frontu, dąży do zawarcia p...",1.277778
214152,We`ll examine it.,Zbadajmy go.,1.500000
214157,What do you think you`re doing?,Wstawaj! Co ty robisz?,1.500000
214161,Where`s your hand grenade?,Gdzie masz swój granat?,1.000000
214163,Let `em have it!,Niech je poczują!,1.333333
...,...,...,...
214943,We don`t want to hate one another.,Bez nienawiści ani pogardy.,1.750000
214944,There`s room for everyone.,Każdy ma swe miejsce.,1.000000
214947,"Greed has poisoned men`s souls, has barricaded...","Chciwość zatruła nasze dusze, wzniosła mury ni...",1.333333
214963,You don`t hate.,Przestańcie nienawidzieć.,1.500000


In [88]:
df_8["eng_text"] = df_8["eng_text"].str.replace("`", "'", regex=False)
df_8["pol_text"] = df_8["pol_text"].str.replace("`", "'", regex=False)

In [95]:
df_8["eng_text"] = df_8["eng_text"].str.replace("~", "", regex=False)
df_8["pol_text"] = df_8["pol_text"].str.replace("~", "", regex=False)

#### 2.11 Del rows with specific case

In [216]:
def del_row_with_char(df, eng_col, pol_col, chars):
    df_proc = df.copy()
    for char in chars:
        mask1 = df_proc[eng_col].apply(lambda x: char in x)
        mask2 = df_proc[pol_col].apply(lambda x: char in x)
        df_proc = df_proc[~(mask1 | mask2)].reset_index(drop=True)
    return df_proc
    

chars = "=+*@#;|_}{"
df_9 = del_row_with_char(df_8, "eng_text", "pol_text", chars)

#### 2.12 Del rows with dialog -

In [217]:
mask1 = df_9["eng_text"].apply(lambda x: bool(re.search(r' +-[^ ]+', x)))
mask2 = df_9["pol_text"].apply(lambda x: bool(re.search(r' +-[^ ]+', x)))
df_data = df_9[~(mask1 | mask2)].reset_index(drop=True)

#### 3. Data Tokenization

In [287]:
uniq_cases = set("".join(df_data["eng_text"].astype(str)))
tab1 = sorted(list(uniq_cases), reverse=True)
print(tab1)

['z', 'y', 'x', 'w', 'v', 'u', 't', 's', 'r', 'q', 'p', 'o', 'n', 'm', 'l', 'k', 'j', 'i', 'h', 'g', 'f', 'e', 'd', 'c', 'b', 'a', ']', '[', 'Z', 'Y', 'X', 'W', 'V', 'U', 'T', 'S', 'R', 'Q', 'P', 'O', 'N', 'M', 'L', 'K', 'J', 'I', 'H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', '?', ':', '9', '8', '7', '6', '5', '4', '3', '2', '1', '0', '/', '.', '-', ',', ')', '(', "'", '%', '$', '"', '!', ' ']


In [288]:
uniq_cases = set("".join(df_data["pol_text"].astype(str)))
tab2 = sorted(list(uniq_cases), reverse=True)
print(tab2)

['ż', 'Ż', 'ź', 'Ź', 'ś', 'Ś', 'ń', 'Ń', 'ł', 'Ł', 'ę', 'Ę', 'ć', 'Ć', 'ą', 'Ą', 'ó', 'Ó', 'z', 'y', 'x', 'w', 'v', 'u', 't', 's', 'r', 'q', 'p', 'o', 'n', 'm', 'l', 'k', 'j', 'i', 'h', 'g', 'f', 'e', 'd', 'c', 'b', 'a', ']', '[', 'Z', 'Y', 'X', 'W', 'V', 'U', 'T', 'S', 'R', 'Q', 'P', 'O', 'N', 'M', 'L', 'K', 'J', 'I', 'H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', '?', ':', '9', '8', '7', '6', '5', '4', '3', '2', '1', '0', '/', '.', '-', ',', ')', '(', "'", '%', '$', '"', '!', ' ']


In [421]:
def tokenize_eng(snt):
    snt = snt.lower()
    snt = re.sub("(?<! )'(?! )", r" '", snt)
    snt = re.sub(r"""([\]?:)\[(/\.\-,%\$"!])""", r" \1 ", snt)
    snt = re.sub(r"\s+", " ", snt).strip()
    return re.split(r'\s+', snt.strip())

In [419]:
def tokenize_pol(snt):
    snt = snt.lower()
    snt = re.sub(r"""([\]?:)\[(/\.\-,%\$"!])""", r" \1 ", snt)
    snt = re.sub(r"\s+", " ", snt).strip()
    return re.split(r'\s+', snt.strip())

In [425]:
df_data = df_data.drop(['len_ratio'], axis=1)

#### 3.1 Splitting and trimming on quantile

In [428]:
df_data['eng_split'] = df_data['eng_text'].apply(tokenize_eng)
df_data['pol_split'] = df_data['pol_text'].apply(tokenize_pol)
df_data.sample(10)

,eng_text,pol_text,eng_split,pol_split
194378,"Chuck, here's a grand from Moore.","Chuck, tu masz kasę od Moore'a.","[chuck, ,, here, 's, a, grand, from, moore, .]","[chuck, ,, tu, masz, kasę, od, moore'a, .]"
286480,Do you suppose he'll listen about Capt. Jack?,"Myślisz, że mnie wysłucha?","[do, you, suppose, he, 'll, listen, about, cap...","[myślisz, ,, że, mnie, wysłucha, ?]"
386683,"What'd you wish, Mary?",O czym pomyślałaś?,"[what, 'd, you, wish, ,, mary, ?]","[o, czym, pomyślałaś, ?]"
276449,Do I look all right?,Jak wyglądam?,"[do, i, look, all, right, ?]","[jak, wyglądam, ?]"
85599,Have you seen a ghost?,Zobaczyłeś ducha?,"[have, you, seen, a, ghost, ?]","[zobaczyłeś, ducha, ?]"
376404,"This way, please.","Tędy, proszę.","[this, way, ,, please, .]","[tędy, ,, proszę, .]"
108087,Good evening.,Dobry wieczór.,"[good, evening, .]","[dobry, wieczór, .]"
392987,Tell him the queen is dead.,Powiedz mu że królowa nie żyje.,"[tell, him, the, queen, is, dead, .]","[powiedz, mu, że, królowa, nie, żyje, .]"
193192,"Now, we'll fix this for you.",Poprawimy to dla ciebie.,"[now, ,, we, 'll, fix, this, for, you, .]","[poprawimy, to, dla, ciebie, .]"
9130,What'd he get in my way for?,To po co mi wchodzi w drogę?,"[what, 'd, he, get, in, my, way, for, ?]","[to, po, co, mi, wchodzi, w, drogę, ?]"


In [452]:
thres1 = df_data['eng_split'].str.len().quantile(0.99)
mask1 = df_data['eng_split'].str.len() > thres1

thres2 = df_data['pol_split'].str.len().quantile(0.99)
mask2 = df_data['pol_split'].str.len() > thres2
print(mask1.sum(), mask2.sum(), (mask1 | mask2).sum())

df_data = df_data[~(mask1 | mask2)].reset_index(drop=True)

4113 3960 5510


#### 3.2 BPE Init. Uniq_freq

In [467]:
uniq_freq_eng = Counter(df_data['eng_split'].explode())
uniq_freq_pol = Counter(df_data['pol_split'].explode())

In [476]:
min_freq = 1
sum(v for k, v in uniq_freq_eng.items() if v <= min_freq)

14144

In [479]:
print(len([k for k, v in uniq_freq_pol.items() if v <= min_freq]))

53192


In [472]:
len(uniq_freq_eng), uniq_freq_eng

(40087,
 Counter({'.': 435802,
          ',': 177291,
          'you': 134548,
          'i': 122767,
          'the': 89859,
          '?': 84640,
          'to': 73192,
          'a': 61661,
          "'s": 57045,
          'it': 55074,
          "'t": 50411,
          'that': 43020,
          'and': 41124,
          'of': 39514,
          'in': 32142,
          '!': 30794,
          'me': 29977,
          'what': 26868,
          'is': 26378,
          'he': 26149,
          'for': 23935,
          'we': 23370,
          'this': 20366,
          'your': 20158,
          'be': 20137,
          'my': 20105,
          "'ll": 20011,
          'have': 19830,
          'don': 18888,
          'do': 18663,
          'on': 18052,
          'no': 17694,
          "'m": 16935,
          'all': 16776,
          'but': 16067,
          'was': 15781,
          "'re": 15483,
          'with': 15435,
          'can': 15365,
          'here': 15059,
          'not': 14896,
          'are': 14431,
 

In [473]:
len(uniq_freq_pol), uniq_freq_pol

(113603,
 Counter({'.': 413780,
          ',': 183948,
          '?': 83444,
          'nie': 82694,
          'to': 53190,
          'się': 47857,
          'w': 33288,
          '!': 29675,
          'na': 29617,
          'i': 28962,
          'że': 27810,
          'z': 26468,
          'tak': 24059,
          'jest': 23868,
          'co': 23351,
          'do': 21754,
          'o': 16448,
          'jak': 16207,
          'mnie': 14640,
          'ale': 14296,
          'mi': 12232,
          'a': 11634,
          'pan': 11472,
          'za': 10597,
          'go': 9391,
          'czy': 9083,
          '"': 8717,
          'ci': 8337,
          'tu': 7998,
          'ja': 7969,
          'pani': 7754,
          'może': 7525,
          'tego': 7476,
          'cię': 7454,
          'tym': 7440,
          'mam': 7102,
          'jestem': 7090,
          'dla': 6931,
          'ma': 6643,
          'już': 6606,
          'tylko': 6593,
          'jeśli': 6583,
          'proszę':

In [ ]:
df_data = 

In [420]:
tokenize_pol('"To jedynie piąta klasa." Och, Juan, kiedy się pobierzemy?')

['"',
 'to',
 'jedynie',
 'piąta',
 'klasa',
 '.',
 '"',
 'och',
 ',',
 'juan',
 ',',
 'kiedy',
 'się',
 'pobierzemy',
 '?']

In [416]:
df_data['pol_text'].sample(30).iloc[10]

'"To jedynie piąta klasa." Och, Juan, kiedy się pobierzemy?'

In [300]:
test_snt2 = "Hmmm, 'cause?"
tokenize_eng(test_snt2)

"hmmm , 'cause ?"

In [221]:
mask = df_data["eng_text"].apply(lambda x: bool(re.search(r"[^ ]+'[^ ]+", x)))
df_data[mask]

,eng_text,pol_text,len_ratio
1,"Yeah, I want to tell him I'm okay.","Tak, powiem, że wszystko w porządku.",1.333333
4,"Elizabeth Keen is not your wife, she's your ta...","Elizabeth Keen nie jest twoją żoną, tylko celem.",1.125000
9,You'll be in America by morning.,Rano będziesz w USA.,1.500000
10,"We're taking you to Washington, D.C.",Zabieramy cię do Waszyngtonu.,1.500000
13,"Yes, we'll take care of it.",Dobrze. Zajmiemy się tym.,1.500000
...,...,...,...
466174,You haven't had a hint of a sneeze for over an...,Nie kichnąłeś już od godziny.,2.400000
466175,I'll bet you're sorry I'm not sneezing.,"Założę się, że się z tego cieszysz.",1.000000
466178,"You're tired, so am I.",Jesteś zmęczony i ja też.,1.000000
466181,What's the matter with...,Co jest z tą...,1.000000


In [289]:
wanted_case = '"'
mask = df_data["eng_text"].apply(lambda x: wanted_case in x)
df_data[mask]

,eng_text,pol_text,len_ratio
185,Because the only word I know that contains the...,"Bo jedyne znane mi słowo z literami ""v"" i ""g"" ...",1.166667
373,"""All that do so are abomination unto the Lord ...","""Te, które to uczynią, obrażą imię Pana, Boga ...",1.222222
494,I will be to this generation a second Mohammed...,"/Będę dla naszego pokolenia /drugim Mahometem,...",1.312500
543,"they would say, ""Shed my blood that I may be s...","""Przeleje swoją krew dla własnego ocalenia i m...",1.454545
628,"My father says, ""Judge not, lest ye be judged.""","Mój ojciec mawia: ""Nie osądzaj, żeby nie zosta...",1.000000
...,...,...,...
461740,"""Willie Stark.","""Willie Stark.",1.000000
461757,"""Voice in the wilderness.","""Wołający na puszczy.",1.333333
461856,"""My friends....","""Przyjaciele...",2.000000
462098,"""Clown, show-off, playboy,"" they yelled out.","""Klaun, playboy"", krzyczeli.",2.000000
